# HICAR 20 m national four-season readiness analysis

Companion notebook for the canonical technical report. It reads only reviewed, local result files and does not submit or modify simulations.

## tl;dr

The executed cell below displays the analyst-reviewed readiness conclusion. No conclusion is stored in the notebook template.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

report_dir = Path(os.environ.get(
    'HICAR_READINESS_REPORT_DIR', 'analysis/hicar_readiness_20m'
)).resolve()
inputs_path = Path(os.environ.get(
    'HICAR_READINESS_INPUTS', str(report_dir / 'inputs.json')
)).resolve()
sys.path.insert(0, str(report_dir))
from build_artifact import derive_datasets, load_evidence

bundle = load_evidence(inputs_path)
datasets = derive_datasets(bundle)
assessment = bundle['reviewed_assessment']
display(Markdown(
    f"**Reviewed status: {assessment['readiness_status'].replace('_', ' ')}.** "
    + assessment['technical_summary']
))

## Context & Methods

The four cases are climatological-season samples, not a climatology. HICAR and native REA-L-CH1 are compared with quality-controlled SwissMetNet values for seven headline metrics: elevation-adjusted 2 m temperature, 2 m relative humidity, elevation-adjusted surface pressure, interval precipitation, snow height, 10 m wind speed, and wind vector. Station-season comparisons retain only positive, equal model pair counts. Primary seasonal summaries weight every eligible station available in that season equally; the exact four-season key intersection is a separate population sensitivity. Lead-hour values retain the upstream evaluator's pooled-pair definition, use elapsed time from the evaluation-window start, and preserve physical simulation lead separately. Negative normalized RMSE differences favor HICAR. Daylight global shortwave is shown separately as HICAR versus SwissMetNet because staged native REA-L has no comparable field.

### Key Assumptions

- The upstream evaluator owns exact valid-time pairing and quality control.
- The station network is the evaluation population; it is not an area-weighted sample of Swiss terrain.
- Station eligibility and valid-pair counts are metric- and season-specific; the station-key union is not a denominator.
- Evaluation lead remains coupled to each event's valid-time sequence; lead 0 initializes the REA-L interval baseline and is not scored. The builder derives and verifies the common positive lead range from the matched endpoint counts (1--168 for the current seven-day evaluation windows).
- Footprint means diagnose point-to-grid representativeness and do not replace the accepted nearest-cell score.

## Data

### 1. Confirm evidence and denominators

In [ ]:
coverage = bundle['national_summary']['coverage']
evidence_overview = pd.DataFrame({
    'quantity': [
        'station-key union',
        'four-season station intersection',
        'station-season-metric rows',
        'restart transitions attested',
        'seasonal metric-season summaries',
        'wind elevation/terrain strata',
    ],
    'value': [
        coverage['station_key_union_count'],
        coverage['station_key_four_season_intersection_count'],
        bundle['national_summary']['station_season_row_count'],
        bundle['restart_transition_audit']['validated_transition_count'],
        len(datasets['seasonal_metrics']),
        len(datasets['elevation_counts']),
    ],
})
evidence_overview

In [ ]:
source_inventory = []
for name, value in bundle['relative'].items():
    if name == 'footprint_reports':
        source_inventory.extend(
            {'source': f'footprint_{season}', 'relative_path': path}
            for season, path in value.items()
        )
    else:
        source_inventory.append({'source': name, 'relative_path': value})
pd.DataFrame(source_inventory)

## Results

### 2. Compare all six seasonal headline metrics

In [ ]:
seasonal = pd.DataFrame(datasets['seasonal_metrics'])
seasonal_pivot = seasonal.pivot(index='season', columns='metric_label', values='normalized_rmse_difference').reindex(['DJF', 'MAM', 'JJA', 'SON'])
axis = seasonal_pivot.plot.bar(figsize=(11, 5), edgecolor='#1f1f1f')
axis.axhline(0, color='#4b4b48', linewidth=1)
axis.set(title='Seasonal skill difference across six metrics', xlabel='Season', ylabel='Normalized RMSE difference')
axis.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
axis.grid(axis='y', color='#d9d9d6', linewidth=0.6)
axis.set_axisbelow(True)
plt.tight_layout()
plt.show()
display(seasonal[['season', 'metric_label', 'unit', 'paired_station_count', 'hicar_rmse', 'rea_l_rmse', 'hicar_bias', 'rea_l_bias', 'hicar_mae', 'rea_l_mae', 'hicar_centered_rmse', 'rea_l_centered_rmse', 'hicar_model_mean', 'rea_l_model_mean', 'hicar_observation_mean', 'rea_l_observation_mean']])
pd.DataFrame(datasets['seasonal_population_sensitivity'])[['season', 'metric_label', 'population', 'paired_station_count', 'hicar_rmse', 'rea_l_rmse', 'normalized_rmse_difference']]

### 3. Inspect within-event lead-hour structure

In [ ]:
lead = pd.DataFrame(datasets['lead_metrics'])
ridge_lead = pd.DataFrame(datasets['ridge_lead_metrics'])
palette = {'DJF': '#1473e6', 'MAM': '#7a9a01', 'JJA': '#d97706', 'SON': '#c2417a'}
figure, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, sharey=True)
metric_group_count = lead['metric_label'].nunique()
for axis, (metric, metric_rows) in zip(axes.flat, lead.groupby('metric_label', sort=False)):
    for season, rows in metric_rows.groupby('season', sort=False):
        axis.plot(rows['lead_hour'], rows['normalized_rmse_difference'], marker='o', markersize=3, label=season, color=palette[season])
    axis.axhline(0, color='#4b4b48', linewidth=1)
    axis.set_title(metric)
    axis.grid(color='#d9d9d6', linewidth=0.6)
for axis in axes.flat[metric_group_count:]:
    axis.axis('off')
axes[0, 0].legend(title='Season', ncol=2)
figure.supxlabel('Lead hour (confounded with valid time and event evolution)')
figure.supylabel('Normalized RMSE difference')
plt.tight_layout()
plt.show()
lead.groupby(['season', 'metric_label', 'unit']).agg(lead_hours=('lead_hour', 'count'), min_pair_count=('pair_count', 'min'), max_pair_count=('pair_count', 'max'), min_difference=('normalized_rmse_difference', 'min'), max_difference=('normalized_rmse_difference', 'max'))

In [ ]:
def slope_per_hour(rows, field):
    x = rows['lead_hour'].astype(float)
    y = rows[field].astype(float)
    centered = x - x.mean()
    return (centered * (y - y.mean())).sum() / (centered ** 2).sum()

lead_trends = []
for (season, metric, unit), rows in lead.groupby(['season', 'metric_label', 'unit'], sort=False):
    rows = rows.sort_values('lead_hour')
    fit_rows = rows[rows['lead_hour'] < 24]
    window = max(1, len(fit_rows) // 3)
    lead_trends.append({
        'season': season, 'metric': metric, 'unit': unit,
        'lead_hours_fitted': len(fit_rows), 'lead_24_excluded_from_slope': True,
        'minimum_pair_count': int(fit_rows['pair_count'].min()),
        'hicar_rmse_slope_per_hour': slope_per_hour(fit_rows, 'hicar_rmse'),
        'rea_l_rmse_slope_per_hour': slope_per_hour(fit_rows, 'rea_l_rmse'),
        'skill_difference_slope_per_hour': slope_per_hour(fit_rows, 'normalized_rmse_difference'),
        'late_minus_early_skill_difference': (
            fit_rows['normalized_rmse_difference'].tail(window).mean()
            - fit_rows['normalized_rmse_difference'].head(window).mean()
        ),
    })
display(pd.DataFrame(lead_trends))
display(lead[lead['lead_hour'].between(11, 14) & lead['hicar_bias'].notna()][['season', 'metric_label', 'lead_hour', 'pair_count', 'hicar_rmse', 'hicar_bias', 'hicar_mae', 'hicar_centered_rmse', 'rea_l_rmse', 'rea_l_bias', 'rea_l_mae', 'rea_l_centered_rmse']])
ridge_lead[ridge_lead['lead_hour'].between(11, 14)][['season', 'stratum_label', 'lead_hour', 'segment', 'pair_count', 'hicar_rmse', 'rea_l_rmse', 'normalized_rmse_difference']]


### 4. Examine elevation and station dependence

In [ ]:
stations = pd.DataFrame(datasets['station_wind'])
figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for axis, (metric, metric_rows) in zip(axes, stations.groupby('metric_label', sort=False)):
    for season, rows in metric_rows.groupby('season', sort=False):
        axis.scatter(rows['station_elevation_m'], rows['normalized_rmse_difference'], s=24, alpha=0.72, label=season, color=palette[season], edgecolors='#1f1f1f', linewidths=0.25)
    axis.axhline(0, color='#4b4b48', linewidth=1)
    axis.set(title=metric, xlabel='Station elevation (m)')
    axis.grid(color='#d9d9d6', linewidth=0.6)
axes[0].set_ylabel('Normalized RMSE difference')
axes[0].legend(title='Season', ncol=2)
plt.tight_layout()
plt.show()
display(stations.nlargest(12, 'normalized_rmse_difference')[['season', 'metric_label', 'station_key', 'station_elevation_m', 'hicar_elevation_m', 'hicar_minus_station_elevation_m', 'nearest_cell_distance_km', 'elevation_class', 'terrain_class', 'terrain_relative_elevation_m', 'pair_count', 'normalized_rmse_difference']])
pd.DataFrame(datasets['elevation_counts'])[['season', 'metric_label', 'stratification', 'stratum_label', 'paired_station_count', 'pair_count_total', 'mean_terrain_relative_elevation_m', 'equal_station_network_hicar_rmse', 'equal_station_network_rea_l_rmse', 'mean_station_hicar_rmse', 'mean_station_rea_l_rmse']]

### 5. Quantify wind-footprint sensitivity

In [ ]:
footprints = pd.DataFrame(datasets['footprint_wind'])
figure, axis = plt.subplots(figsize=(6, 6))
radius_colors = {0.4: '#1473e6', 1.0: '#b68400'}
for radius, rows in footprints.groupby('radius_km', sort=True):
    axis.scatter(rows['nearest_rmse_m_s'], rows['footprint_mean_rmse_m_s'], s=32, alpha=0.75, label=f'{radius:g} km', color=radius_colors[float(radius)], edgecolors='#1f1f1f', linewidths=0.3)
limits = [0, max(footprints['nearest_rmse_m_s'].max(), footprints['footprint_mean_rmse_m_s'].max()) * 1.05]
axis.plot(limits, limits, color='#4b4b48', linestyle='--', linewidth=1, label='Equal RMSE')
axis.set(xlim=limits, ylim=limits, title='Nearest-cell versus footprint-mean wind RMSE', xlabel='Nearest-cell RMSE (m s$^{-1}$)', ylabel='Footprint-mean RMSE (m s$^{-1}$)')
axis.legend()
axis.grid(color='#d9d9d6', linewidth=0.6)
plt.tight_layout()
plt.show()
footprints.sort_values('footprint_minus_nearest_rmse_m_s', ascending=False).head(15)

## Takeaways

The following statements come from `reviewed_assessment.json` after the analyst has inspected the executed evidence above.

In [ ]:
for key in ('seasonal_skill', 'lead_time', 'elevation_wind', 'footprint_sensitivity', 'inputs_and_grid', 'restart'):
    finding = assessment['findings'][key]
    display(Markdown(f"### {finding['heading']}\n\n{finding['body']}"))
display(Markdown('### Limitations\n\n' + '\n'.join(f"- {item}" for item in assessment['limitations'])))
display(Markdown('### Recommended next steps\n\n' + '\n'.join(f"- {item}" for item in assessment['recommended_next_steps'])))
display(Markdown('### Further questions\n\n' + '\n'.join(f"- {item}" for item in assessment['further_questions'])))